# Ciência de Dados em Python — formulário preliminar (Estácio)

**Objetivo:** localizar rapidamente conceitos em questões objetivas e substituir números nos códigos para resolver cálculos. **Versão 0.1:** ajustar com feedback dos exercícios reais.

**Como usar:** Ctrl+F para procurar palavras-chave; execute as células de código alterando apenas os dados de entrada. Dependências: `numpy` (opcional: `scikit-learn`); cálculos principais usam Python e NumPy.

**Índice:** 1 Introdução · 2 CNN · 3 Visão computacional · 4 PLN · 5 Reconhecimento de voz · Glossário.

## 1. Introdução

- **IA:** campo amplo de sistemas que executam tarefas associadas à inteligência.
- **Machine learning:** aprende padrões a partir de dados; **deep learning:** redes neurais com múltiplas camadas.
- **Supervisionado:** exemplos rotulados (classificação/regressão); **não supervisionado:** sem rótulos (agrupamento); **reforço:** recompensas por ações.
- **Treino:** ajusta parâmetros; **validação:** escolhe hiperparâmetros/monitora generalização; **teste:** avalia desempenho final em dados não vistos.
- **Overfitting:** bom desempenho no treino, ruim fora dele; mitigação: regularização, dropout, aumento de dados, early stopping, mais dados.
- **Parâmetros** (pesos/vieses) são aprendidos; **hiperparâmetros** (taxa de aprendizado, batch size, momentum) são configurados.
- **Normalização/padronização:** ajustar escala das entradas; estimar transformações no treino e aplicar nos demais conjuntos sem vazamento.

## 2. Redes neurais convolucionais (CNN)

### Operação de convolução

Filtro/kernel percorre a matriz, multiplica elementos correspondentes e soma. **A maioria das bibliotecas de CNN calcula correlação cruzada** (sem inverter kernel), embora chame de convolução. *Stride* = passo; *padding* = borda adicionada; *pooling* = redução espacial (max/average).

Dimensão de saída (por eixo):

$$O=\left\lfloor\frac{I+2P-K}{S}\right\rfloor+1$$

- I: tamanho da entrada; P: padding de cada lado; K: tamanho do kernel; S: stride. Fórmula para dilatação 1.
- **ReLU:** $f(x)=\max(0,x)$; **sigmoid:** saída entre 0 e 1; **softmax:** probabilidades por classe que somam 1; **tanh:** entre −1 e 1.
- **Loss/função de erro:** mede discrepância entre previsão e alvo (ex.: entropia cruzada, erro quadrático). **ReLU não é função de erro**; lambda é apenas recurso sintático Python, não um tipo de loss. Loss calculada sobre previsões e rótulos durante treinamento; validação também monitora loss sem atualizar pesos.
- **Backpropagation:** calcula gradientes; **otimizador** atualiza pesos sinápticos e vieses.
- **Learning rate:** tamanho do passo de atualização; pode variar com agendamentos (*schedulers*).
- **Momentum:** acumula direção de atualizações anteriores, suavizando oscilações e frequentemente acelerando convergência.
- **Fine-tuning:** continuar treinamento de modelo pré-treinado, ajustando parte ou todos os pesos para nova tarefa.
- **Overfitting:** nenhum otimizador garante prevenção; dropout, data augmentation, weight decay e early stopping ajudam.
- **Pré-processamento de imagem:** redimensionamento, normalização, canais de cor, divisão treino/validação/teste, augmentação somente no treino.
- **Predição:** logits = escores brutos; sigmoid para binário/multirrótulo; softmax para classes mutuamente exclusivas; argmax = classe mais provável, não certeza.

### Arquiteturas

| Modelo | Identificação em prova |
|---|---|
| LeNet | reconhecimento de dígitos, CNN histórica |
| AlexNet | avanço no ImageNet de 2012, ReLU e GPUs |
| VGGNet | pilhas de convoluções 3×3, arquitetura profunda e relativamente simples |
| GoogLeNet/Inception | módulos Inception, múltiplos tamanhos de filtros em paralelo, convoluções 1×1 |
| ResNet | conexões residuais/atalhos (skip connections) |

### Cálculo automático de correlação cruzada 2D

In [ ]:
import numpy as np

def aplicar_filtro(matriz, kernel, stride=1, padding=0, inverter_kernel=False):
    x = np.asarray(matriz, dtype=float)
    k = np.asarray(kernel, dtype=float)
    if inverter_kernel:  # convolução matemática verdadeira
        k = np.flip(k)
    x = np.pad(x, padding)
    h, w = k.shape
    saida = [[np.sum(x[i:i+h, j:j+w] * k)
              for j in range(0, x.shape[1]-w+1, stride)]
             for i in range(0, x.shape[0]-h+1, stride)]
    return np.array(saida)

matriz = [[1,2,3],[4,5,6],[7,8,9]]  # ALTERE
kernel = [[1,0],[0,-1]]               # ALTERE
print(aplicar_filtro(matriz, kernel, stride=1, padding=0))
print('Convolução com kernel invertido:', aplicar_filtro(matriz, kernel, inverter_kernel=True))

In [ ]:
import math
I, K, P, S = 32, 3, 1, 1  # ALTERE para cada dimensão
O = math.floor((I + 2*P - K)/S) + 1
print('Tamanho da saída:', O)

## 3. IA aplicada à visão computacional

- **Visão computacional:** extrair informações de imagens/vídeos. **OpenCV (cv2):** ler imagens, transformar, filtrar, detectar bordas, processar vídeo; não é, por si, um modelo de CNN.
- **Classificação:** rótulo para imagem inteira. **Detecção:** classes e caixas delimitadoras (*bounding boxes*).
- **Segmentação semântica:** classe para cada pixel; duas pessoas podem receber o mesmo rótulo 'pessoa'. **Segmentação de instâncias:** separa cada pessoa individualmente. **Panóptica:** combina ambas.
- **CNN:** classificação, detecção, segmentação e extração de características.
- **GAN:** gerador produz amostras e discriminador tenta distinguir real de gerado; aplicações em síntese/aumento de imagens.
- **Supervisionado:** usa imagens com rótulos; segmentação supervisionada costuma exigir máscaras rotuladas.

### Matriz de confusão (binária)

| | Previsto positivo | Previsto negativo |
|---|---:|---:|
| Real positivo | VP (TP) | FN |
| Real negativo | FP | VN (TN) |

- **Acurácia:** $(VP+VN)/(VP+VN+FP+FN)$.
- **Precisão (precision):** $VP/(VP+FP)$ — entre os previstos positivos, quantos eram positivos?
- **Recall/sensibilidade:** $VP/(VP+FN)$ — entre os positivos reais, quantos foram encontrados?
- **Especificidade:** $VN/(VN+FP)$.
- **F1:** $2\cdot precision\cdot recall/(precision+recall)$.
- Em classes desbalanceadas, acurácia pode enganar. **FP** é falso alarme; **FN** é positivo perdido.

In [ ]:
VP, VN, FP, FN = 40, 50, 10, 5  # ALTERE

def dividir(a,b): return a/b if b else float('nan')
acuracia = dividir(VP+VN, VP+VN+FP+FN)
precisao = dividir(VP, VP+FP)
recall = dividir(VP, VP+FN)
especificidade = dividir(VN, VN+FP)
f1 = dividir(2*VP, 2*VP+FP+FN)
for nome, valor in [('Acurácia',acuracia),('Precisão',precisao),('Recall',recall),('Especificidade',especificidade),('F1',f1)]:
    print(f'{nome}: {valor:.4f} ({valor:.2%})')

## 4. Machine learning aplicado ao processamento de linguagem natural (PLN)

| Ferramenta | Reconheça pelo uso |
|---|---|
| spaCy | tokenização, lematização, entidades nomeadas, análise sintática; pipelines de PLN |
| Gensim | Word2Vec, Doc2Vec, modelagem de tópicos (LDA) |
| scikit-learn | TF-IDF, classificação, clustering, métricas, modelos tradicionais |
| Transformers (Hugging Face) | carregar modelos pré-treinados BERT/GPT, tokenizadores, pipelines |
| TensorFlow / Keras | desenvolver e treinar redes neurais; também PLN |
| PyTorch | treinamento de redes neurais e modelos modernos de PLN |

| Arquitetura/modelo | Principal característica |
|---|---|
| RNN | processa sequências recorrendo a estado oculto; dificuldades com dependências longas |
| LSTM/GRU | RNNs com mecanismos de controle de memória |
| BERT | Transformer encoder, contexto bidirecional; classificação, NER, extração |
| GPT | Transformer decoder autoregressivo; prevê próximo token, geração de texto |
| Transformer | atenção (self-attention), paralelização no treinamento; base de BERT/GPT |

**Tokenização:** separar texto em tokens, nem sempre palavras completas. **Lematização:** reduzir palavra à forma canônica ('correndo' → 'correr'); **stemming:** cortar afixos sem garantir palavra real. **Embeddings:** vetores numéricos que representam tokens/palavras. **TF-IDF:** peso estatístico de termos em documentos.

**Atenção à questão:** frameworks são ferramentas, arquiteturas são estruturas de modelos e BERT/GPT são famílias de modelos; categorias não são intercambiáveis.

## 5. IA aplicada ao reconhecimento de voz

### Pipeline clássico de reconhecimento automático de fala (ASR)

1. **Captura/digitalização:** microfone → sinal analógico → amostragem temporal e quantização de amplitude → amostras digitais.
2. **Pré-processamento:** redução de ruído, normalização, recorte e extração de características (ex.: MFCC).
3. **Modelo acústico:** associa características sonoras a fonemas/unidades acústicas.
4. **Modelo léxico / dicionário:** mapeia palavras às sequências de fonemas/pronúncias.
5. **Modelo de linguagem:** atribui probabilidades a sequências de palavras (ex.: n-gramas).
6. **Decodificador:** combina evidências e escolhe transcrição provável.

**Atenção:** modelos end-to-end podem integrar etapas em uma única rede; a divisão acima descreve sistemas clássicos.

- **Grafema:** unidade de escrita (letra/grupo de letras). **Fonema:** unidade distintiva de som. Relação não é sempre 1:1.
- **speech_recognition:** nome do pacote Python usado para interface de reconhecimento de fala (`SpeechRecognition`); **NumPy:** arrays; **Matplotlib:** gráficos; **Tkinter:** interface gráfica. `voice_recognition` não é o nome padrão desse pacote.
- **Lematização:** forma de dicionário das palavras, não é o mesmo que reconhecimento acústico.
- **Unigrama** (n=1): P(palavra); **bigrama** (n=2): P(palavra atual | anterior); **trigrama** (n=3): P(atual | duas anteriores).
- **Perplexidade:** mede quão surpreendente é a sequência para o modelo; **menor é melhor** quando a avaliação é comparável (mesma tokenização e dados).
- **Sinal digital:** frequência de amostragem em Hz = amostras por segundo; resolução em bits determina níveis de quantização (idealmente $2^b$). **Erro de quantização:** diferença entre valor original e representado.
- **Pares ordenados (tempo, amplitude):** pontos de uma forma de onda amostrada. Se questão pedir erro total, identificar se é soma de erros absolutos, quadráticos ou com sinal.

### N-gramas: identificar e calcular probabilidades

Estimativa sem suavização de um bigrama: $P(w_i\mid w_{i-1})=C(w_{i-1},w_i)/C(w_{i-1})$.

Para trigrama: $P(w_i\mid w_{i-2},w_{i-1})=C(w_{i-2},w_{i-1},w_i)/C(w_{i-2},w_{i-1})$.

**Atenção:** contagens devem corresponder ao mesmo contexto de previsão; em aplicações reais são comuns tokens de início/fim e suavização.

In [ ]:
from collections import Counter
frases = ['eu gosto de python', 'eu gosto de dados', 'eu estudo python']  # ALTERE
n = 2  # 1=unigramas, 2=bigramas, 3=trigramas
contagens = Counter(tuple(tokens[i:i+n]) for frase in frases
                    for tokens in [frase.lower().split()]
                    for i in range(len(tokens)-n+1))
print(contagens)
# Exemplo: P('gosto' | 'eu') = contagem ('eu','gosto') / contagem contextos 'eu'
contexto = ('eu',)
alvo = 'gosto'
numerador = contagens[contexto+(alvo,)]
denominador = sum(v for seq,v in contagens.items() if seq[:-1] == contexto)
print('Probabilidade:', numerador/denominador if denominador else 'contexto ausente')

### Perplexidade

Para probabilidades condicionais $p_1,\ldots,p_N$ dos tokens efetivamente observados:

$$PP=\exp\left(-\frac{1}{N}\sum_{i=1}^N\ln p_i\right)$$

Equivale a $\left(\prod_i p_i\right)^{-1/N}$. Probabilidade zero sem suavização implica perplexidade infinita.

In [ ]:
import math
probabilidades = [0.5, 0.25, 0.5]  # ALTERE: probabilidade de cada palavra observada
if any(p < 0 or p > 1 for p in probabilidades) or not probabilidades:
    raise ValueError('Informe probabilidades em (0,1]')
pp = math.inf if 0 in probabilidades else math.exp(-sum(math.log(p) for p in probabilidades)/len(probabilidades))
print('Perplexidade:', pp)

### Matriz de estados / cadeias de Markov

Se **linhas = estado atual** e **colunas = próximo estado**, a entrada $P[i,j]$ é a probabilidade de ir de i para j em um passo. Linhas somam 1.

Distribuição após $n$ passos: $\pi_n=\pi_0P^n$ (vetor linha). Em questões com orientação oposta (colunas = atual), a interpretação muda.

In [ ]:
import numpy as np
P = np.array([[0.8, 0.2], [0.3, 0.7]])  # ALTERE
estado_atual = 0  # índice 0 = estado 1
proximo_estado = 1  # índice 1 = estado 2
passos = 1
assert np.allclose(P.sum(axis=1), 1), 'Cada linha deve somar 1'
print('Transição em um passo:', P[estado_atual, proximo_estado])
print(f'Transição em {passos} passo(s):', np.linalg.matrix_power(P, passos)[estado_atual, proximo_estado])

### Erros entre sinal original e digitalizado

Se a questão apresentar pares (valor real, valor aproximado), verifique qual medida solicita: erro absoluto total $\sum|x_i-\hat x_i|$; erro quadrático total $\sum(x_i-\hat x_i)^2$; erro médio quadrático $MSE=\frac1N\sum(x_i-\hat x_i)^2$. **Não confundir** com taxa de erro de palavras (WER) no ASR.

In [ ]:
import numpy as np
original = np.array([1.0, 2.0, 3.0])  # ALTERE
aproximado = np.array([1.1, 1.8, 3.2])  # ALTERE
erros = original - aproximado
print('Erro com sinal (soma):', erros.sum())
print('Erro absoluto total:', np.abs(erros).sum())
print('Erro quadrático total:', np.square(erros).sum())
print('Erro médio quadrático (MSE):', np.square(erros).mean())

## Glossário e pegadinhas rápidas

- **ReLU** = ativação; **cross-entropy/MSE** = loss; **SGD/Adam** = otimizadores.
- **Momentum** altera dinâmica do gradiente; **learning rate** altera tamanho do passo.
- **VGG** = filtros 3×3; **GoogLeNet** = Inception; **ResNet** = conexões residuais.
- **Precision** = VP/(VP+FP); **recall** = VP/(VP+FN); **accuracy** = (VP+VN)/total.
- **Segmentação semântica** identifica classes por pixel; **instâncias** distinguem objetos da mesma classe.
- **spaCy** = pipeline linguístico; **Gensim** = tópicos/embeddings; **scikit-learn** = ML clássico; **Transformers** = modelos pré-treinados.
- **BERT** = compreensão bidirecional; **GPT** = geração autoregressiva; **RNN** = sequência recorrente.
- **Modelo acústico** = som; **léxico** = pronúncia; **linguagem** = sequência de palavras.
- **n-grama** = sequência de n tokens; **perplexidade** = medida de incerteza preditiva.
- **P[i,j]** = probabilidade de i→j, se matriz estocástica por linhas.

### Fontes de consulta

- Google ML Crash Course: https://developers.google.com/machine-learning/crash-course
- Stanford CS231n: https://cs231n.github.io/convolutional-networks/
- Hugging Face NLP Course: https://huggingface.co/learn/llm-course/chapter1/1
- OpenCV Python: https://docs.opencv.org/4.x/d6/d00/tutorial_py_root.html
- CMUSphinx: https://cmusphinx.github.io/wiki/models/

**Próxima revisão:** registrar questões que não forem resolvidas apenas com este notebook, corrigir lacunas e adicionar exemplos específicos.